# Prepare Forward-Filled Data from Oanda Data and Save to InfluxDB

Weekends are skipped in this output.

## Load useful libraries

In [ ]:
from influxdb_client import InfluxDBClient

In [ ]:
from forex.etl.config.database_config import INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG, INFLUXDB_BUCKET
from forex.etl.pipelines.ForwardFillInator import ForwardFillInator
from forex.etl.config.schema_config import schema_measurement_name_ff, ALLOWED_TAGS_ff, ALLOWED_FIELDS_ff
from python_tools_and_shortcuts.databases.influxdb.InfluxDbTool import InfluxDbTool

## User settings

In [ ]:
instrument = 'EUR/USD'
granularity = 'H1'

In [ ]:
lookback = 604800 + 1  # seconds

## Open database connection

In [ ]:
ifc = InfluxDbTool(INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG)

## Define a function to retrieve the latest timestamp in a measurement

In [ ]:
def get_latest_timestamp(url, token, org, bucket, schema_measurement_name, instrument, granularity):
    query = f'''
            from(bucket: "{bucket}")
              |> range(start: 0)
              |> filter(fn: (r) => r._measurement == "{schema_measurement_name}")
              |> filter(fn: (r) => r.granularity == "{granularity}")
              |> filter(fn: (r) => r.instrument == "{instrument}")
              |> keep(columns: ["_time"])
              |> group()
              |> sort(columns: ["_time"], desc: true)
              |> limit(n: 1)
              |> map(fn: (r) => ({{ r with unix_epoch_s: int(v: uint(v: r._time) / uint(v: 1000000000)) }}))
              |> keep(columns: ["unix_epoch_s"])
        '''

    with InfluxDBClient(url=url, token=token, org=org) as client:
        tables = client.query_api().query(query=query, org=org)
        
    for table in tables:
        for record in table.records:
            return record['unix_epoch_s']

## Figure out the most recent timestamp in for the measurement

In [ ]:
last_timestamp = None
try:
    last_timestamp = get_latest_timestamp(INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG, INFLUXDB_BUCKET, schema_measurement_name_ff, instrument, granularity)
except:
    pass

## Create the class that performs the heavy lifting

In [ ]:
if last_timestamp == None:
    ebpd = ForwardFillInator(instrument, granularity, ifc, cutoff_timestamp = 0)
else:
    #
    # We'll perhaps want to evenually refactor the lookback operation to improve speed:
    #
    ebpd = ForwardFillInator(instrument, granularity, ifc, cutoff_timestamp = last_timestamp - lookback)

## Create the forward-filled dataframe

In [ ]:
ebpd.pull_data()
ebpd.fit()        
df_ff = ebpd.df_all_time_diff_market_open_forward_filled
df_ff['unix_epoch_s'] = df_ff['unix_epoch_s'].astype('Int64')
df_ff.sort_values(by = ['unix_epoch_s'], inplace = True)

df_ff['instrument'] = instrument
df_ff['granularity'] = granularity

df_ff = df_ff[
    [
        'instrument', 'granularity', 'unix_epoch_s',
        'volume',
        'ask_open', 'ask_high', 'ask_low', 'ask_close',
        'mid_open', 'mid_high', 'mid_low', 'mid_close',
        'bid_open', 'bid_high', 'bid_low', 'bid_close',
        'spread_open', 'spread_high', 'spread_low', 'spread_close',
    ]
]

## Convert to a format suitable for database insert

In [ ]:
to_influx_list = []
for r in df_ff.to_dict(orient = 'records'):
    record_result = {'measurement' : schema_measurement_name_ff, 'fields' : {}, 'tags' : {}}
    for key in r.keys():
        key_str = str(key) # not sure this is necessary

        # make these lists variables
        if key_str in ALLOWED_TAGS_ff:
            record_result['tags'][key_str] = r[key_str]

        elif key_str in ALLOWED_FIELDS_ff:
            record_result['fields'][key_str] = r[key_str]

        elif key_str in ['unix_epoch_s']:
            record_result['time'] = r[key_str]

        else:
            pass
            # we should throw an error here
    to_influx_list.append(record_result)

## Insert results into the database

In [ ]:
ifc.insert_dictionary_list(
    to_influx_list,
    ALLOWED_TAGS_ff,
    ALLOWED_FIELDS_ff,
    INFLUXDB_BUCKET,
)

## Close database connection

In [ ]:
del(ifc)